In [ ]:
import torch
from datasets import load_from_disk
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, 
    BitsAndBytesConfig, TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import shutil

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

shutil.copytree(
    "/kaggle/input/datasets/karanveertinna/amazon-reviews/augmentated_amazon_reviews_full/train",
    "/kaggle/working/train",
    dirs_exist_ok=True
)

dataset_path = "/kaggle/working/train"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = 512

model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    quantization_config=bnb_config, 
    device_map="auto"
)

lora_config = LoraConfig(
    r=8, 
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

def format_prompts(example):
    bias_text = "Biased" if example["bias_label"] == 1 else "Unbiased"

    return (
        f"<|system|>\nSummarize and detect bias.</s>\n"
        f"<|user|>\n{example['review_body']}</s>\n"
        f"<|assistant|>\nSummary: {example['summary']} | Bias: {bias_text}</s>"
    )

dataset = load_from_disk(dataset_path)

training_args = TrainingArguments(
    output_dir="/kaggle/tinyllama-amazon-tuned",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=500, 
    # fp16=True,
    bf16=True,
    save_total_limit=2,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    formatting_func=format_prompts, 
)

print("Starting Fine-Tuning...")
trainer.train()
trainer.save_model("/kaggle/working/final_model")
print("Training Complete! Model saved to ./final_model")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting Fine-Tuning...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such a

Step,Training Loss
10,2.305356
20,1.663511
30,1.398054
40,1.340595
50,1.318960
60,1.330229
70,1.328203
80,1.326845
90,1.297939
100,1.285172


Training Complete! Model saved to ./final_model


In [1]:
!pip install -U bitsandbytes>=0.46.1 trl